In [1]:
import json

# Replace 'your_file.json' with your file path
with open('confidence_foo_model_0.json', 'r') as f:
    data = json.load(f)

# Now `data` is a Python dictionary (or list, depending on the JSON structure)
print(data.keys())

dict_keys(['confidence_score', 'ptm', 'iptm', 'ligand_iptm', 'protein_iptm', 'complex_plddt', 'complex_iplddt', 'complex_pde', 'complex_ipde', 'chains_ptm', 'pair_chains_iptm'])


In [5]:
%matplotlib qt

import numpy as np
from pickle import load
from matplotlib import pyplot as plt

def read_pkl_boltz(pkl_file):
    """
    Reads the distograms pkl file outputed from Alphafold and returns the bins (63) and distograms (NxNx64)
    """
    with open(pkl_file, "rb") as f:
        data_first = load(f)
        print(data_first.keys())
    
    data = data_first['distogram']
    # Bins -> distance axis: initially 64 bins from 2 to 22 but the last one covers also above 22.
    # 20/64 = 0.3125 = delta_bins. first bin (2.3125) covers 2 to 2.3125. 
    # center of it is therefore 2.3125 - delta_bins/2
    print(data.keys())
    bins = data['bin_edges']
    print(bins)
    print(bins.shape)
    delta_bins = bins[1]-bins[0]
    # Shift the bins such that the distribution data point are assumed in the center of the bin:
    # (And converting the bin values from Angstrom to nm)
    #bins = [(el-(delta_bins/2))*0.1 for el in bins]
    bins = (bins - (delta_bins/2))*0.1
    distograms = data['softmax']
    print(distograms[0, 0].sum())
    print(distograms.shape)

    contacts = np.zeros((1, 1, 1, 64), dtype=distograms.dtype)
    contacts[:, :, :, :20] = 1.0
    prob_contact = (distograms * contacts).sum(-1)[0]

    print(prob_contact.shape)
    print(data['prob_contact'].shape)
    print(prob_contact[10, 10])
    print(data['prob_contact'][10, 10])

    fig, ax = plt.subplots(1, 2)
    ax[0].imshow(prob_contact, cmap="RdYlBu_r")
    ax[1].imshow(data['prob_contact'], cmap="RdYlBu_r")
    

    return bins, distograms

out = read_pkl_boltz('distogram_8r1f__A1_Q05086--8r1f__B1_P03126_model_0.pkl')

dict_keys(['distogram'])
dict_keys(['softmax', 'bin_edges', 'prob_contact'])
[ 2.         2.3225806  2.6451612  2.967742   3.2903225  3.612903
  3.935484   4.2580643  4.580645   4.903226   5.225806   5.548387
  5.870968   6.193548   6.516129   6.8387094  7.16129    7.483871
  7.8064513  8.129032   8.451612   8.774194   9.096774   9.419354
  9.741936  10.064516  10.387096  10.709677  11.032258  11.354838
 11.677419  12.        12.322581  12.645162  12.967742  13.290323
 13.612904  13.935484  14.258064  14.580646  14.903226  15.225806
 15.548388  15.870968  16.193548  16.516129  16.83871   17.161291
 17.483871  17.806452  18.129032  18.451612  18.774193  19.096775
 19.419355  19.741936  20.064516  20.387096  20.709677  21.032259
 21.35484   21.67742   22.       ]
(63,)
1.0
(1033, 1033, 64)
(1033, 1033)
(1033, 1033)
5.8402942e-24
5.8402942e-24
